In [1]:
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Literal

In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("API_TOKEN"),
    base_url="https://openrouter.ai/api/v1"
)

In [4]:
@tool
def calculator(expression: str) -> str:
    """Calculate a math expression. Use for any arithmetic. Args: expression: e.g., '25 * 47'"""
    try:
        return f"{expression} = {eval(expression)}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def search_web(query: str) -> str:
    """Search the web for information. Use for current events or facts. Args: query: search terms"""
    return f"Search results for '{query}': Several major developments reported this week."


In [5]:
tools = [calculator, search_web]
llm_with_tools = llm.bind_tools(tools)


tools_by_name = {t.name: t for t in tools}

In [6]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [7]:
def llm_node(state: State) -> dict:
    """
    The 'thinking' node. The LLM looks at the conversation and decides:
    - Should I call a tool? → generates a tool_call
    - Should I respond directly? → generates a text response
    """
    system = SystemMessage(
        content="You are a helpful assistant. Use tools when needed to get accurate information."
    )
    
    response = llm_with_tools.invoke([system] + state["messages"])
    
    return {"messages": [response]}   


def tool_node(state: State) -> dict:
    """
    The 'acting' node. Executes whatever tools the LLM requested
    and returns the results as ToolMessages.
    """
    results = []
    
    
    last_message = state["messages"][-1]
    
   
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]              
        tool_args = tool_call["args"]              
        

        tool_result = tools_by_name[tool_name].invoke(tool_args)
        
        
        results.append(ToolMessage(
            content=str(tool_result),             
            tool_call_id=tool_call["id"]          
        ))
    
    return {"messages": results} 

In [8]:
def should_continue(state: State) -> Literal["tool_node", "__end__"]:
    """
    After the LLM responds, check: did it call any tools?
    If YES → go to tool_node to execute them
    If NO  → go to END (the LLM gave a final text answer)
    """
    last_message = state["messages"][-1]
    
    
    if last_message.tool_calls:
        return "tool_node"
    
    return "__end__"

In [9]:
graph_builder = StateGraph(State)


graph_builder.add_node("llm_node", llm_node)      
graph_builder.add_node("tool_node", tool_node)    


graph_builder.add_edge(START, "llm_node")          

graph_builder.add_conditional_edges(
    "llm_node",                                      
    should_continue,                                  
    ["tool_node", "__end__"]                          
)

graph_builder.add_edge("tool_node", "llm_node")    


agent = graph_builder.compile()

In [11]:
result = agent.invoke({
    "messages": [HumanMessage(content="Who is the US President?")]
})


for msg in result["messages"]:
    if hasattr(msg, 'content') and msg.content:
        print(f"[{msg.__class__.__name__}]: {msg.content}\n")

[HumanMessage]: Who is the US President?

[ToolMessage]: Search results for 'current US President 2023': Several major developments reported this week.

[AIMessage]: As of 2023, the current President of the United States is Joe Biden.

